# Conditional Edge

## 조건부 Edge(`Conditional Edge`) 사용해 분기 만들기
* `Conditional Edge`를 이용해 상황에 따라 다른 노드로 연결되도록 구현할 수 있습니다.

In [ ]:
from typing import Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage, HumanMessage

class AgentState(BaseModel):
    # Annotated[타입, 메타데이터] -> 해당 변수에 '메타데이터'에 해당하는 특성(기능)을 부여함
    # add_messages -> message에 적용하는 메타데이터, Node에서 history에 추가할 항목만 반환해도 알아서 리스트에 추가해 줌
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list) # 맨 처음 생성 시 빈 리스트로 시작
    last_player: Literal['A', 'B']
    number: int
    end: bool

In [ ]:
def node_A(state: AgentState) -> dict:
    num = state.number
    new_num = num ** 2
    return {
        'chat_history': [HumanMessage(name='A', content=f"숫자를 {num}에서 {new_num}(으)로 바꿈")],
        'number': new_num,
        'last_player': 'A'
    }

In [ ]:
def node_B(state: AgentState) -> dict:
    num = state.number
    new_num = num - 1
    return {
        'chat_history': [HumanMessage(name='B', content=f"숫자를 {num}에서 {new_num}(으)로 변경")],
        'number': new_num,
        'last_player': 'B'
    }

In [ ]:
def node_judge(state: AgentState) -> dict:
    end = True if state.number > 100 else False
    return {
        'chat_history': [HumanMessage(name='judge', content=f"최종 숫자 {state.number}(으)로 종료합니다" if end else "계속 진행하세요")],
        'end': end
    }

In [ ]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(AgentState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('judge', node_judge)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'judge')
workflow.add_edge('B', 'judge')

## `route` 만들고 `conditional edge`에 부여하기
* `route`는 조건에 따라서 달라지는 목적지 `node`명 을 반환하는 함수입니다
* `workflow.add_conditional_edges`함수에 `route`를 적용하면, route가 반환한 `node`명으로 라우팅됩니다.

In [ ]:
def judge_route(state: AgentState):
    # end가 True면 END로,
    # 짝수면 A로, 홀수면 B로 가는 route
    if state.end:
        return END
    elif state.last_player == 'B':
        return 'A'
    else:
        return 'B'

workflow.add_conditional_edges(
    'judge', 
    judge_route
)

In [ ]:
app = workflow.compile()

init_input = AgentState(chat_history=[], last_player='A', number=2, end=False)

for chunk in app.stream(init_input):
    print(chunk)

## 토마토 게임 만들기
* ai와 토마토 게임을 진행할 수 있는 LangGraph 기반 게임을 만들어 봅시다
* 규칙: 두 명이 번갈아가면서 '토마토'를 한글자씩 말합니다. 올바르게 말하지 못하면 패배합니다.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class TomatoGameState(BaseModel):
    next_player: Literal['user', 'ai']
    ...

In [ ]:
def user_node(state: TomatoGameState) -> dict:
    user_input = input("당신의 대답: ")
    return {
        'player_input': user_input
    }

def ai_node(state: TomatoGameState) -> dict:
    ...

def judge_node(state: TomatoGameState) -> dict:
    ...